In [1]:
import boto3
import os
from flask import Flask, jsonify, request
from flask_cors import CORS
from datetime import datetime, timedelta
import time
from data_generation import generate_cell_tower_data, generate_predicted_outages
import uuid
from utils import n_towers_within_zone
from dotenv import load_dotenv
from decimal import Decimal


In [2]:
load_dotenv()

# --- AWS DynamoDB setup ---
dynamodb = boto3.resource(
    "dynamodb",
    region_name="us-east-1", # change region if needed
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY"), # load from env variables
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)

# Change this to your DynamoDB table name
cell_towers_table = dynamodb.Table("cell-towers")
outages_table = dynamodb.Table("outages")

bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

In [3]:
# clear table
cell_towers_table.scan()['Items']
for tower_id in cell_towers_table.scan()['Items']:
    response = cell_towers_table.delete_item(Key={'tower_id': tower_id['tower_id']})

len(cell_towers_table.scan()['Items'])

0

In [4]:
cell_tower_data = generate_cell_tower_data(num_towers=300)

for idx, tower_id in enumerate(cell_tower_data['tower_id']):
    item = {
        'tower_id': tower_id,
        'status': cell_tower_data['status'][idx],
        'longitude': Decimal(str(cell_tower_data['longitude'][idx])),
        'latitude': Decimal(str(cell_tower_data['latitude'][idx])),
        'signal_strength': Decimal(str(cell_tower_data['signal_strength'][idx])),
        'coverage_radius': Decimal(str(cell_tower_data['coverage_radius'][idx])),
        'bandwidth': cell_tower_data['bandwidth'][idx],
        'technology': cell_tower_data['technology'][idx],
    }
    response = cell_towers_table.put_item(Item=item)


In [5]:
len(cell_towers_table.scan()['Items'])

300

In [6]:
cell_towers_table.scan()['Items'][:3]

[{'technology': '5G',
  'coverage_radius': Decimal('7.345253768717736'),
  'signal_strength': Decimal('-81.81165501166731'),
  'tower_id': 'FN-1183',
  'status': 'Active',
  'longitude': Decimal('-102.37846815287591'),
  'latitude': Decimal('28.65929051830793'),
  'bandwidth': Decimal('100')},
 {'technology': '5G',
  'coverage_radius': Decimal('9.52587816156644'),
  'signal_strength': Decimal('-88.06651420007017'),
  'tower_id': 'FN-1021',
  'status': 'Active',
  'longitude': Decimal('-100.24816443209897'),
  'latitude': Decimal('29.045259681298685'),
  'bandwidth': Decimal('100')},
 {'technology': '4G LTE',
  'coverage_radius': Decimal('1.629384683291751'),
  'signal_strength': Decimal('-60.8442756510296'),
  'tower_id': 'FN-1184',
  'status': 'Active',
  'longitude': Decimal('-106.03122215439427'),
  'latitude': Decimal('33.22012432110988'),
  'bandwidth': Decimal('80')}]

In [7]:
# clear table
outages_table.scan()['Items']
for outage in outages_table.scan()['Items']:
    response = outages_table.delete_item(Key={'outage_id': outage['outage_id']})

len(outages_table.scan()['Items'])

0

In [8]:
outage_data = generate_predicted_outages(10)
cell_tower_data = cell_towers_table.scan()['Items']

for idx, outage_id in enumerate(outage_data['outage_id']):
    item = {
        'outage_id': outage_id,
        'event': outage_data['event'][idx],
        'severity': outage_data['severity'][idx],
        'center_longitude': Decimal(str(outage_data['center_longitude'][idx])),
        'center_latitude': Decimal(str(outage_data['center_latitude'][idx])),
        'radius': Decimal(str(outage_data['radius'][idx]))
    }
    
    # get the number of towers within the zone
    n_towers, n_towers_down = n_towers_within_zone(outage_data['center_latitude'][idx], outage_data['center_longitude'][idx], outage_data['radius'][idx], cell_tower_data)
    
    item['towers_total'] = n_towers 
    item['towers_affected'] = n_towers_down
    
    response = outages_table.put_item(Item=item)


In [9]:
len(outages_table.scan()['Items'])

10

In [10]:
outages_table.scan()['Items'][:5]

[{'center_latitude': Decimal('35.256223516367555'),
  'event': 'Flood',
  'towers_affected': Decimal('0'),
  'towers_total': Decimal('0'),
  'radius': Decimal('13.758741853338542'),
  'center_longitude': Decimal('-103.5458966167979'),
  'outage_id': 'Outage-5',
  'severity': 'High'},
 {'center_latitude': Decimal('33.36064305276954'),
  'event': 'Flood',
  'towers_affected': Decimal('0'),
  'towers_total': Decimal('0'),
  'radius': Decimal('6.971169460425295'),
  'center_longitude': Decimal('-105.67037567064573'),
  'outage_id': 'Outage-7',
  'severity': 'Low'},
 {'center_latitude': Decimal('29.888784397520524'),
  'event': 'Cyber Attack',
  'towers_affected': Decimal('0'),
  'towers_total': Decimal('0'),
  'radius': Decimal('20.953668565844076'),
  'center_longitude': Decimal('-94.45205514496817'),
  'outage_id': 'Outage-1',
  'severity': 'High'},
 {'center_latitude': Decimal('26.441773478876495'),
  'event': 'Flood',
  'towers_affected': Decimal('0'),
  'towers_total': Decimal('1'),
 